In [ ]:
library(abind)
library(IRdisplay)
set.seed(1)
if (getwd() != "/projects/ps-gymreklab/amassara/simulate_gwas") {
    setwd("/projects/ps-gymreklab/amassara/simulate_gwas")
}
out = "temp/susieR"
dir.create(out, showWarnings = FALSE)

In [ ]:
n = 500
p = 1000
b = rep(0,p)
b[200] = 1
b[800] = 1
X = matrix(rnorm(n*p),nrow=n,ncol=p)
X[,200] = X[,400]
X[,600] = X[,800]
y = X %*% b + rnorm(n)

In [ ]:
pdf('truth.pdf', width =5, height = 5, pointsize=16)
plot(b, col="black", pch=16, main = 'True effect size')
pos = 1:length(b)
points(pos[b!=0],b[b!=0],col=2,pch=16)
dev.off()

In [ ]:
display_pdf(file = 'truth.pdf')

In [ ]:
# alpha = 1
# y.fit = glmnet::glmnet(X,y,alpha = alpha,intercept = FALSE)
# y.cv  = glmnet::cv.glmnet(X,y,alpha = alpha,intercept = FALSE,
#                          lambda = y.fit$lambda)
# bhat  = glmnet::predict.glmnet(y.fit,type ="coefficients",
#                                s = y.cv$lambda.min)[-1,1]

In [ ]:
# pdf('lasso.pdf', width =5, height = 5, pointsize=16)
# plot(bhat, col="black", pch=16, main = 'Lasso')
# pos = 1:length(bhat)
# points(pos[b!=0],bhat[b!=0],col=2,pch=16)   
# dev.off()

In [ ]:
# display_pdf('lasso.pdf')

# Existing Bayesian methods for sparse regression

In [ ]:
mm_regression = function(X, Y, Z=NULL) {
  if (!is.null(Z)) {
      Z = as.matrix(Z)
  }
  reg = lapply(seq_len(ncol(Y)), function (i) simplify2array(susieR:::univariate_regression(X, Y[,i], Z)))
  reg = do.call(abind, c(reg, list(along=0)))
  # return array: out[1,,] is betahat, out[2,,] is shat
  return(aperm(reg, c(3,2,1)))
}
sumstats = mm_regression(as.matrix(X), as.matrix(y))
dat = list(X=X,Y=as.matrix(y))
input = paste0(out,'/Toy.sumstats.rds')
saveRDS(list(data=dat, sumstats=sumstats), input)

In [ ]:
output = paste0(out, "/Toy.N2finemapping.FINEMAP")
args = "--n-causal-snps 2"
commandArgs = function(...) 1

# source(paste0(.libPaths(), '/susieR/code/finemap.R'))
source("/projects/ps-gymreklab/amassara/simulate_gwas/temp/finemap.R")

In [ ]:
finemap = readRDS(paste0(out,"/Toy.N2finemapping.FINEMAP.rds"))[[1]]
head(finemap$set)

In [ ]:
snp = finemap$snp
pip = snp[order(as.numeric(snp$snp)),]$snp_prob

In [ ]:
pdf(paste0(out,'/Toy.finemap.pdf'), width =5, height = 5, pointsize=16)
susieR::susie_plot(pip, y='PIP', b=b, main = 'Bayesian sparse regression')
dev.off()

In [ ]:
display_pdf(file=paste0(out,'/Toy.finemap.pdf'))

# SuSiE

In [ ]:
fitted = susieR::susie(X, y, L=5,
               estimate_residual_variance=TRUE, 
               scaled_prior_variance=0.2,
               tol=1e-3, track_fit=TRUE, min_abs_corr=0.1)

In [ ]:
pdf(paste0(out,'/Toy.susie.pdf'), width =5, height = 5, pointsize=16)
susieR::susie_plot(fitted, y='PIP', b=b, max_cs=0.4, main = paste('SuSiE, ', length(fitted$sets$cs), 'CS identified'))
dev.off()

In [ ]:
display_pdf(file=paste0(out,'/Toy.susie.pdf'))

## SuSiE Effect Size Estimate

In [ ]:
bhat = coef(fitted)[-1]
pdf(paste0(out,'/Toy.susie_eff.pdf'), width =5, height = 5, pointsize=16)
susieR::susie_plot(bhat, y='bhat', b=b, main = 'SuSiE, effect size estimate') 
dev.off()

In [ ]:
display_pdf(file=paste0(out,'/Toy.susie_eff.pdf'))